In [1]:
import torch
import torch.nn.functional as F

class SimpleCausalAttentionWithCache(torch.nn.Module):
    def __init__(self, d_model=8):
        super().__init__()
        self.d_model = d_model
        # Projections for a single head
        self.q_proj = torch.nn.Linear(d_model, d_model, bias=False)
        self.k_proj = torch.nn.Linear(d_model, d_model, bias=False)
        self.v_proj = torch.nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x, kv_cache=None):
        """
        x: (Batch, 1, d_model) - Only the newest single token is passed in during generation!
        kv_cache: Tuple of (past_k, past_v) tensors
        """
        # 1. Project the new token input
        q = self.q_proj(x) # (Batch, 1, d_model)
        k = self.k_proj(x) # (Batch, 1, d_model)
        v = self.v_proj(x) # (Batch, 1, d_model)
        
        # 2. Manage the Key-Value Cache
        if kv_cache is not None:
            past_k, past_v = kv_cache
            # Concatenate the new single step key/value along the sequence dimension (dim=1)
            k = torch.cat([past_k, k], dim=1)
            v = torch.cat([past_v, v], dim=1)
            
        new_kv_cache = (k, v) # Save updated tensors back to cache
        
        # 3. Compute Attention using the single Query and all compiled Keys
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_model ** 0.5)
        # Note: No causal mask needed here because Q is only 1 token long (it cannot see the future)
        
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, v)
        
        return output, new_kv_cache

# =====================================================================
# THE TOP-P NUCLEUS SAMPLING FILTER
# =====================================================================
def sample_top_p(logits, top_p=0.9, temperature=0.7):
    # Scale by temperature
    logits = logits / temperature
    
    # Sort logits descending
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    
    # Identify indices to remove (cumulative probability is above top_p threshold)
    sorted_indices_to_remove = cumulative_probs > top_p
    # Shift right to keep the very first token that breached the threshold boundary
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0
    
    # Mask out filtered indices by forcing their logit score to negative infinity
    indices_to_remove = sorted_indices_to_remove.scatter(dim=-1, index=sorted_indices, src=sorted_indices_to_remove)
    logits[indices_to_remove] = float('-inf')
    
    # Sample from the filtered distribution
    probabilities = F.softmax(logits, dim=-1)
    next_token = torch.multinomial(probabilities, num_samples=1)
    return next_token

# =====================================================================
# SIMULATED INFERENCE PIPELINE RUNTIME
# =====================================================================
torch.manual_seed(42)
attention_block = SimpleCausalAttentionWithCache(d_model=8)
lm_head = torch.nn.Linear(8, 50) # Map model outputs to 50 vocabulary options

# Initialize our dynamic tracking cache as clean empty states
kv_cache = None
# Simulate an active single incoming feature vector token input block
current_token_embeddings = torch.randn(1, 1, 8) 

print("--- Launching Cached Autoregressive Generation Engine ---")
for step in range(3):
    # Pass ONLY the single new token vector and the persistent historical VRAM cache
    context_vector, kv_cache = attention_block(current_token_embeddings, kv_cache=kv_cache)
    
    # Generate vocabulary logits for this isolated position step
    step_logits = lm_head(context_vector[:, -1, :])
    
    # Filter choices via Top-P nucleus thresholds and return token ID
    next_token_id = sample_top_p(step_logits, top_p=0.85, temperature=0.8)
    
    print(f"Generation Step {step+1} Completed Success Track")
    print(f" -> Retained Key Cache VRAM Shape Vector Size: {kv_cache[0].shape} (Batch, Seq_Len, Dim)")
    print(f" -> Sampled Token ID Selected: {next_token_id.item()}\n")
    
    # Mock lookup: pull the embedding coordinates for the newly chosen token to drive the next loop step
    current_token_embeddings = torch.randn(1, 1, 8)

--- Launching Cached Autoregressive Generation Engine ---
Generation Step 1 Completed Success Track
 -> Retained Key Cache VRAM Shape Vector Size: torch.Size([1, 1, 8]) (Batch, Seq_Len, Dim)
 -> Sampled Token ID Selected: 23

Generation Step 2 Completed Success Track
 -> Retained Key Cache VRAM Shape Vector Size: torch.Size([1, 2, 8]) (Batch, Seq_Len, Dim)
 -> Sampled Token ID Selected: 27

Generation Step 3 Completed Success Track
 -> Retained Key Cache VRAM Shape Vector Size: torch.Size([1, 3, 8]) (Batch, Seq_Len, Dim)
 -> Sampled Token ID Selected: 35

